# Adaptive RAG — Real Embeddings Upgrade (bge-small-en-v1.5)

This notebook runs the same `rag_system` project you have locally, but swaps the
CPU-only TF-IDF fallback embedder for a real transformer bi-encoder
(`BAAI/bge-small-en-v1.5`) via `sentence-transformers`, and prints a
side-by-side comparison of retrieval hit rate, routing accuracy, and
faithfulness between the two backends.

**Steps:**
1. Runtime is fine as CPU-only (no GPU needed — bge-small is small enough).
2. Upload `rag_system.zip` when prompted (the same one from your local project —
   zip your `rag_system` folder before running this).
3. Run all cells. The last cell prints the comparison table you can paste into
   your resume writeup / project README.

In [1]:
!pip install -q sentence-transformers rank_bm25 scikit-learn

In [2]:
from google.colab import files
print("Upload rag_system.zip (zip your local rag_system folder and upload it here):")
uploaded = files.upload()

Upload rag_system.zip (zip your local rag_system folder and upload it here):


Saving rag_system.zip to rag_system.zip


In [3]:
import zipfile, os

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

os.chdir('rag_system')
print("Now in:", os.getcwd())
!ls

Now in: /content/rag_system
app.py	colab_embeddings_upgrade.ipynb	data  README.md  requirements.txt  src


## Run the comparison

`--backend both` runs the full eval suite once with TF-IDF and once with the
real bge-small-en-v1.5 embeddings, then prints a side-by-side table. This is
the exact same `src/eval.py` file from your local project — nothing was
duplicated for Colab, so any changes you make locally (e.g. adding harder
eval questions or expanding the corpus) will show up here automatically the
next time you re-zip and re-upload.

In [4]:
!python -m src.eval --backend both

######################################################################
# EMBEDDER BACKEND: tfidf
######################################################################
RETRIEVAL ABLATION: dense-only vs BM25-only vs hybrid (RRF)
[dense-only] Retrieval hit_rate@5: 95.24% (20/21)
[bm25-only] Retrieval hit_rate@5: 100.00% (21/21)
[hybrid-rrf] Retrieval hit_rate@5: 100.00% (21/21)

QUERY ROUTING
  [OK ] 'What error rate triggers an automatic rollback?...' expected=simple got=simple (sim=0.764)
  [OK ] 'How many approvals does a change to the payments service nee...' expected=simple got=simple (sim=0.780)
  [OK ] 'If a canary release for a high-traffic service breaches the ...' expected=multi_hop got=multi_hop (sim=0.890)
  [OK ] 'A new engineer wants production database access to debug an ...' expected=multi_hop got=multi_hop (sim=0.694)
  [OK ] 'Who must be notified within one hour if customer data is con...' expected=multi_hop got=multi_hop (sim=0.963)
  [OK ] 'How many days of annual lea

## Router diagnostic: does the single-token-collapse OOS gate transfer to real embeddings?

The local (TF-IDF) router was just upgraded with a second out-of-scope gate:
if removing any ONE content word from the query collapses max corpus
similarity by more than `single_token_collapse_ratio` (default 0.7), the
query is treated as a vocabulary coincidence (e.g. "what's the weather like
today?" scoring 0.95 against a facilities doc that only mentions "severe
weather advisory") rather than a genuine topical match.

That 0.7 default was tuned against TF-IDF+SVD similarity, where unrelated
text pairs score ~0.0. Real bge-small-en-v1.5 embeddings are anisotropic —
this project's own notes already found unrelated text pairs sit at a
similarity floor of ~0.40-0.44, not 0 — so masking a word probably won't
collapse similarity as dramatically, and 0.7 may be the wrong cutoff here.
This cell prints the actual collapse ratio per eval question on the
transformer backend so you can tell whether 0.7 still separates OOS
vocabulary-coincidence queries from genuine in-scope matches, the same way
`default_oos_threshold(backend)` was originally derived.

In [5]:
import json, os
from src.ingestion import load_corpus
from src.embeddings import get_embedder
from src.retrieval import HybridRetriever
from src.router import RuleBasedRouter, default_oos_threshold

corpus_dir = os.path.join("data", "corpus")
eval_items = json.load(open(os.path.join("data", "eval", "eval_set.json"), encoding="utf-8"))

chunks = load_corpus(corpus_dir)
embedder = get_embedder("transformer")
retriever = HybridRetriever(chunks, embedder)
router = RuleBasedRouter(embedder, retriever.chunk_vecs,
                          oos_threshold=default_oos_threshold("transformer"))

print(f"{'expected':>12}  {'max_sim':>8}  {'collapse':>9}  question")
worst_in_scope = 0.0
oos_ratios = []
for item in eval_items:
    q = item["question"]
    max_sim = router._max_sim(q)
    ratio = router._single_token_collapse(q, max_sim)
    if item["type"] == "out_of_scope":
        oos_ratios.append((q, ratio, max_sim))
    elif max_sim >= router.oos_threshold:
        worst_in_scope = max(worst_in_scope, ratio)
    print(f"{item['type']:>12}  {max_sim:8.3f}  {ratio:9.3f}  {q[:60]}")

print()
print(f"Worst (highest) collapse ratio among genuine in-scope queries: {worst_in_scope:.3f}")
print("OOS queries (ratio only meaningful if max_sim already clears oos_threshold):")
for q, ratio, max_sim in oos_ratios:
    print(f"  max_sim={max_sim:.3f}  collapse={ratio:.3f}  {q}")
print()
print("If any OOS query's collapse ratio is close to or below worst_in_scope, "
      "0.7 does not cleanly separate them on this backend -- pick a new "
      "single_token_collapse_ratio partway between the two, add it as the "
      "'transformer' case in a default_collapse_ratio(backend) helper "
      "alongside default_oos_threshold, and report both numbers.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    expected   max_sim   collapse  question
      simple     0.825      0.115  What error rate triggers an automatic rollback?
      simple     0.864      0.060  How many approvals does a change to the payments service nee
   multi_hop     0.908      0.090  If a canary release for a high-traffic service breaches the 
   multi_hop     0.741      0.062  A new engineer wants production database access to debug an 
   multi_hop     0.883      0.043  Who must be notified within one hour if customer data is con
      simple     0.839      0.103  How many days of annual leave can be carried over to the nex
      simple     0.821      0.080  Can an engineer join the on-call rotation in their first wee
out_of_scope     0.433      0.000  What is the capital of France?
      simple     0.832      0.035  Can the company deploy a security patch during the fiscal qu
out_of_scope     0.522      0.078  What's the weather like today?
      simple     0.772      0.056  How soon do employees have to hand

## Interpreting the result

- If `hybrid hit_rate@5` is now higher than `bm25 hit_rate@5` on the paraphrase
  test questions (q11, q12 in `eval_set.json`), that's the real evidence that
  dense retrieval is catching semantic matches BM25 misses — write that
  specific number down.
- If all three still saturate at 100% even with real embeddings, that confirms
  the corpus itself is too small/easy (not a fallback-embedder artifact) — the
  next real step is scaling the corpus (step 3 in your plan), not further
  embedding upgrades.
- `faithfulness` numbers will likely stay similar between backends, since the
  generator/hallucination-checker logic doesn't change — that's a separate
  upgrade path (LLM generation + NLI checking) for later.

### To carry this upgrade back into your local project
Once you're happy with `transformer` backend results, just tell your local
pipeline/eval calls to use `embedder_backend="transformer"` (or `--backend
transformer`) instead of the default `"tfidf"` — no code changes needed,
since `get_embedder()` already handles both. You'll need
`pip install sentence-transformers` locally too, and an internet connection
the first time it downloads the model (~130MB, cached after that).